# 03 — SAM 3 exploration
Verify the SAM 3 architecture (LoRA target paths in the vision encoder) and the
LoRA injection trainable-param counts, on the real gated weights
(`facebook/sam3` via HF `transformers`).

In [1]:
#autoreload, so edits to .py files show without restarting
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.insert(0, "../src")   #repo convention so `medseg` imports (same as 00_data.ipynb)
print(sys.executable)          #sanity check: should be the sam-medseg-mac env

/Users/nikhilclaude/anaconda3/envs/sam-medseg-mac/bin/python


## 1. Load SAM 3 (real gated weights)

In [3]:
from transformers import Sam3Model

#real gated weights — reads from the HF cache filled by `hf download facebook/sam3`
sam3_model = Sam3Model.from_pretrained("facebook/sam3")

/Users/nikhilclaude/anaconda3/envs/sam-medseg-mac/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 22767.82it/s]


## 2. Architecture — LoRA target paths
Expect 32 vision-encoder layers, d=1024, separate `q_proj`/`v_proj` (bias=True) at
`vision_encoder.backbone.layers.{i}.attention.*`. Injection targets the vision
backbone ONLY — text_encoder / geometry_encoder / detr_* / mask_decoder are skipped.

In [4]:
for name, module in sam3_model.named_modules():

    if name.endswith('q_proj') and name.startswith('vision'):
        print(name, module)

vision_encoder.backbone.layers.0.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.1.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.2.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.3.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.4.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.5.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.6.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.7.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.8.attention.q_proj Linear(in_features=1024, out_features=1024, bias=True)
vision_encoder.backbone.layers.9.attention.q_proj Linea

In [5]:
total = sum(p.numel() for p in sam3_model.parameters())
print(f"{total:,} total params")

840,376,374 total params


## 3. Unit tests

In [6]:
!pytest ../tests/test_lora.py -v
!pytest ../tests/test_utils.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- /Users/nikhilclaude/anaconda3/envs/sam-medseg-mac/bin/python
cachedir: .pytest_cache
rootdir: /Users/nikhilclaude/Documents/Projects/DeepLearning/group_project
plugins: anyio-4.14.2
collected 4 items                                                              

../tests/test_lora.py::test_output_shape PASSED                          [ 25%]
../tests/test_lora.py::test_zero_at_init PASSED                          [ 50%]
../tests/test_lora.py::test_param_count PASSED                           [ 75%]
../tests/test_lora.py::test_linear_with_lora PASSED                      [100%]

============================== 4 passed in 0.44s ===============================
============================= test session starts ==============================
platform darwin -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- /Users/nikhilclaude/anaconda3/envs/sam-meds

## 4. LoRA injection — trainable-param verification
Predicted: `32 layers × 2 projections × (2·4·1024) = 524,288` adapter params.

In [7]:
from medseg.models.lora import inject_lora_into_sam3

#fresh load in the SAME cell as the injection, so re-running the cell
#can never inject twice (wrap-inside-a-wrap would double the trainable count)
fresh_model = Sam3Model.from_pretrained("facebook/sam3")
fresh_model = inject_lora_into_sam3(fresh_model, r=4, alpha=4)

trainable = sum(p.numel() for p in fresh_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in fresh_model.parameters())
print(f"{trainable:,} / {total:,} trainable  ({100*trainable/total:.4f}%)")

Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 26626.93it/s]

524,288 / 840,900,662 trainable  (0.0623%)


## 5. Revision pin
HF model repos are git repos — each cache snapshot folder below is named after the
commit hash of the repo state we downloaded. If Meta updates `facebook/sam3`, a bare
`from_pretrained("facebook/sam3")` would silently fetch different weights, breaking
the LoRA-vs-full-finetune comparison with Sean's arm. So both arms should pin the
exact commit:

```python
Sam3Model.from_pretrained("facebook/sam3", revision="<hash below>")
```

Downloaded 2026-07-17: `3c879f39826c281e95690f02c7821c4de09afae7` — proposed to Sean
as the shared pin; goes into `configs/model/sam3.yaml` once agreed.

In [8]:
#each folder here = one downloaded commit of the facebook/sam3 repo
!ls ~/.cache/huggingface/hub/models--facebook--sam3/snapshots/

3c879f39826c281e95690f02c7821c4de09afae7
